- In the previous steps, the atmospheric parameters of this spectrum (Teff, log g, [Fe/H], and vmic) have already been determined and stored in output_atmos_params_dumpfile_path.
- The main purpose of this code is to fix the previously derived, converged atmospheric parameters and compute the abundances of other elements.
- When entering the abundance calculation (EW-based method), the key requirement is to provide ispec.model_spectrum_from_ew with a linemasks file that contains both the cross-matching information and the line-fitting results.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
from scipy.stats import norm
from userlist import *
import re

In [ ]:
# =============== 1. Get paths and read data =================
# Read the normalized spectrum
star_spectrum_file = spectrum_norm_path
star_spectrum = ispec.read_spectrum(star_spectrum_file)

# =============== 2. Read linemasks =================
linemask_output_folder = output_folder + "/linemasks"
# Linemasks containing cross-matching information and line-fitting results,
# required by model_spectrum_from_ew
linemasks = ispec.read_line_regions(
    linemask_output_folder + f"/{target}_melendez2014_star_fitted_linemasks.txt"
)
print(f"Number of lines in the linemask: {len(linemasks)}")

# =============== 3. Read dump file with atmospheric parameters =================
dump_file = output_atmos_params_dumpfile_path.replace(".dump", "_q2.dump")
payload = ispec.restore_results(dump_file)

# Compatibility handling:
# regardless of how many extra items are stored in the dump,
# only the first three (params, errors, status) are used here
params = payload[0]
errors = payload[1]
status_q2 = payload[2]

# =============== 4. Extract atmospheric parameters =================
teff, tefferr = params['teff'], errors['teff']
logg, loggerr = params['logg'], errors['logg']
mh, mherr = params['MH'], errors['MH']
vmic, vmicerr = params['vmic'], errors['vmic']
alpha, alphaerr = params['alpha'], errors['alpha']

print("==================== Reference stellar parameters =================")
print(f"Teff = {initial_teff}, logg = {initial_logg}, MH = {initial_MH}")
print("==================== Fitted stellar parameters =================")
print(f"Teff = {teff:.2f} +/- {tefferr:.2f} K")
print(f"logg = {logg:.2f} +/- {loggerr:.2f} dex")
print(f"[M/H] = {mh:.2f} +/- {mherr:.2f} dex")
print(f"vmic = {vmic:.2f} +/- {vmicerr:.2f} km/s")

In [ ]:
# =============== Plot Gaussian fits (all elements) ===============
base_output_dir = output_folder + "/figs_ele_GaussianFits"
os.makedirs(base_output_dir, exist_ok=True)
deleted_elelines_folder_path = base_output_dir + "/deleted"
modified_elelines_folder_path = base_output_dir + "/modified"
os.makedirs(deleted_elelines_folder_path, exist_ok=True)
os.makedirs(modified_elelines_folder_path, exist_ok=True)

w_range = 0.25  # nm

# Define Gaussian function
def gaussian(x, mu, sig, A, baseline):
    return baseline + A * np.exp(-(x - mu)**2 / (2 * sig**2))

# Loop over all elements
elements = np.unique(linemasks['element'])
for ele in elements:
    if "Fe" in ele:   # Skip iron to avoid duplication
        continue

    ele_lines = linemasks[linemasks['element'] == ele]
    
    if len(ele_lines) == 0:
        continue
    
    # Create subfolder for the current element
    ele_output_dir = os.path.join(base_output_dir, ele.replace(" ", "_"))
    os.makedirs(ele_output_dir, exist_ok=True)

    # Loop over all lines of the current element
    for idx, line in enumerate(ele_lines):
        mu = line['mu']
        sig = line['sig']
        A = line['A']
        baseline = line['baseline']
        
        if sig == 0 or mu == 0:  # Invalid fit
            continue

        # Extract spectral segment
        mask = (
            (star_spectrum['waveobs'] >= mu - w_range) &
            (star_spectrum['waveobs'] <= mu + w_range)
        )
        wave = star_spectrum['waveobs'][mask]
        flux = star_spectrum['flux'][mask]

        # Gaussian fitted curve
        fit_x = np.linspace(mu - w_range, mu + w_range, 300)
        fit_y = gaussian(fit_x, mu, sig, A, baseline)

        # Plot
        plt.figure(figsize=(8, 5), dpi=128)
        plt.plot(wave, flux, label='Observed Spectrum', color='blue', lw=0.7)
        plt.plot(fit_x, fit_y, '--', label='Gaussian Fit', color='red')
        plt.axvline(mu, color='orange', linestyle=':', label=f"$\mu$ = {mu:.3f} nm")
        plt.title(f"Gaussian Fit: {line['element']} {line['wave_A']:.2f} Å", fontsize=16)
        plt.xlabel("Wavelength (nm)", fontsize=16)
        plt.ylabel("Normalized Flux", fontsize=16)
        plt.xticks(fontsize=13)
        plt.yticks(fontsize=13)
        plt.ticklabel_format(style='plain', axis='x')

        # Annotation
        text = (
            f"log(gf) = {line['loggf']:.2f}\n"
            f"EP = {line['lower_state_eV']:.2f} eV\n"
            f"EW = {line['ew']:.1f} mÅ"
        )
        plt.text(
            0.75, 0.05, text,
            transform=plt.gca().transAxes,
            fontsize=13,
            bbox=dict(facecolor='white', alpha=0.8)
        )
        plt.legend(loc="lower left", fontsize=13)
        plt.tight_layout()

        # Save figure
        out_path = os.path.join(
            ele_output_dir,
            f"{ele}_{idx+1}_{line['wave_A']:.2f}.png"
        )
        plt.savefig(out_path)
        plt.close()

    print(f"Element {ele}: plotted {len(ele_lines)} figures, saved in {ele_output_dir}/")

print(f"\nPlotting completed. All element results are saved in {base_output_dir}/")

- Refer to the function determine_abundances_from_ew in `example.py`.
- Atmospheric model grid:
model = `ispec_dir + "/input/atmospheres/MARCS.GES/"`
(a reduced grid with preliminary interpolation applied).
- Solar abundance reference:
`ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"`

In [ ]:
# Filter invalid lines
linemasks = linemasks[linemasks['wave_nm'] > 0]   # successfully cross-matched
linemasks = linemasks[linemasks['ew'] > 0]        # valid EW

# --- Determine abundances from EW using previously fitted lines ---
code = "moog"

# Stellar parameters
teff = teff
logg = logg
MH = mh
alpha = alpha
microturbulence_vel = vmic  # km/s

# ========== Load model atmosphere and solar abundances ==========
# Selected model atmosphere grid and solar abundance reference
# Different grids correspond to different atmosphere libraries
#model = ispec_dir + "/input/atmospheres/MARCS/"     # large grid
model = ispec_dir + "/input/atmospheres/MARCS.GES/" # reduced grid with pre-interpolation
#model = ispec_dir + "/input/atmospheres/MARCS.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.APOGEE/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Castelli/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kurucz/"
#model = ispec_dir + "/input/atmospheres/ATLAS9.Kirby/"

# Solar abundance reference
if "ATLAS" in model:
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.1998/stdatom.dat"
else:
    solar_abundances_file = ispec_dir + "/input/abundances/Grevesse.2007/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2005/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Asplund.2009/stdatom.dat"
#solar_abundances_file = ispec_dir + "/input/abundances/Anders.1989/stdatom.dat"

# ========== Load models ==========
# Load model atmosphere grid
modeled_layers_pack = ispec.load_modeled_layers_pack(model)
# Load solar abundances
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)

# Validate that parameters are within the model grid
if not ispec.valid_atmosphere_target(
    modeled_layers_pack, {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha}
):
    msg = (
        "The specified effective temperature, surface gravity (log g), "
        "and metallicity [M/H] are outside the model atmosphere grid."
    )
    print(msg)

# Prepare interpolated model atmosphere
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {'teff': teff, 'logg': logg, 'MH': MH, 'alpha': alpha},
    code=code
)

## line by line differential

In [ ]:
# ---------- 0) Normalize element labels: convert "Fe 1" -> "Fe I" for consistency with the solar library ----------
def norm_elem(e):
    s = str(e).strip()
    s = s.replace(" 1", " I").replace(" 2", " II")
    return s

# ---------- 1) Read solar_lines_instru (the per-line solar reference produced in step 0_) ----------
solar_lines_csv = os.path.join(ispec_dir, "input", "solar_lines_instru.csv")
sun_master = pd.read_csv(solar_lines_csv)

# ---------- Filter by instrument ----------
sun_inst = sun_master[sun_master["instrument"].astype(str).str.strip() == str(instrument_name).strip()].copy()
if len(sun_inst) == 0:
    raise RuntimeError(f"[export_q2_inputs] solar master has no rows for instrument={instrument_name}")

# ---------- Automatically select the closest resolution ----------
# Cast to float to support int/float/string representations
avail = pd.to_numeric(sun_inst["resolution"], errors="coerce").values
if not np.isfinite(avail).any():
    raise RuntimeError(f"[export_q2_inputs] solar master resolution column cannot be parsed for instrument={instrument_name}")

req = float(from_resolution)
res_use = avail[np.nanargmin(np.abs(avail - req))]
sun_sel = sun_inst

sun_sel["element_norm"] = sun_sel["element"].map(norm_elem)

# ---------- 2) Prepare inputs for iSpec abundance calculation: solar abundances + model atmosphere ----------
# These variables are typically defined earlier (e.g., in userlist.py):
# model, code, solar_abundances_file
solar_abundances = ispec.read_solar_abundances(solar_abundances_file)
modeled_layers_pack = ispec.load_modeled_layers_pack(model)

# Build atmosphere_layers using q2-derived teff/logg/mh/alpha
atmosphere_layers = ispec.interpolate_atmosphere_layers(
    modeled_layers_pack,
    {"teff": teff, "logg": logg, "MH": mh, "alpha": alpha},
    code=code
)

microturbulence_vel = vmic

# ---------- 3) Compute per-line logeps and [X/H] for each element ----------
elem_norm_all = np.array([norm_elem(e) for e in linemasks["element"]])

# If a 'discarded' field exists, filter it out (recommended)
if "discarded" in linemasks.dtype.names:
    disc = linemasks["discarded"]

    # Boolean type
    if disc.dtype == np.bool_:
        mask_use = ~disc

    # Numeric type: 0 = keep, non-zero = discard
    elif np.issubdtype(disc.dtype, np.number):
        mask_use = (disc == 0)

    # String/object type: recognize common truthy flags
    else:
        disc_str = np.array([str(x).strip().lower() for x in disc])
        is_discarded = np.isin(disc_str, ["true", "1", "yes", "y", "t"])
        mask_use = ~is_discarded
else:
    mask_use = np.ones(len(linemasks), dtype=bool)

linemasks_use = linemasks[mask_use]
elem_norm_use = elem_norm_all[mask_use]

elements = np.unique(elem_norm_use)

star_lines = []
for ele in elements:
    sel = (elem_norm_use == ele)
    lm_ele = linemasks_use[sel]
    if len(lm_ele) == 0:
        continue

    spec_abund, normal_abund, x_over_h, x_over_fe = ispec.determine_abundances(
        atmosphere_layers,
        teff, logg, mh, alpha,
        lm_ele,
        solar_abundances,
        microturbulence_vel=microturbulence_vel,
        verbose=0,
        code=code
    )

    spec_abund = np.asarray(spec_abund, dtype=float)  # logeps - 12
    x_over_h   = np.asarray(x_over_h, dtype=float)

    for lm, logeps_m12, xh in zip(lm_ele, spec_abund, x_over_h):
        if not (np.isfinite(logeps_m12) and np.isfinite(xh)):
            continue
        star_lines.append({
            "id": str(target),
            "element": ele,
            "wave_A": float(lm["wave_A"]),
            "EP": float(lm["lower_state_eV"]),
            "loggf": float(lm["loggf"]),
            "ew_mA": float(lm["ew"]),
            "logeps_star": float(logeps_m12 + 12.0),
            "[X/H]_star_Grevesse": float(xh),
        })

df_star_lines = pd.DataFrame(star_lines)
print("star per-line rows:", len(df_star_lines))

# ---------- 4) Merge with the solar per-line library to obtain line-by-line differentials ----------
wave_tol = 0.001  # Å; increase to ~0.005 Å if needed
df_star_lines["wave_key"] = (df_star_lines["wave_A"] / wave_tol).round().astype(int)
sun_sel["wave_key"] = (sun_sel["wave_A"] / wave_tol).round().astype(int)

df_merge = pd.merge(
    df_star_lines,
    sun_sel,
    left_on=["element", "wave_key"],
    right_on=["element_norm", "wave_key"],
    how="inner",
    suffixes=("", "_sun")
)

# Line-by-line differentials (both Δlogeps and Δ[X/H])
df_merge["dlogeps"] = df_merge["logeps_star"] - df_merge["logeps_sun"]
df_merge["d[X/H]"]  = df_merge["[X/H]_star_Grevesse"] - df_merge["[X/H]_sun_Grevesse"]

# Select columns for output
df_merge_out = df_merge[[
    "id","element","wave_A","EP","loggf","ew_mA",
    "logeps_star","logeps_sun","dlogeps",
    "[X/H]_star_Grevesse","[X/H]_sun_Grevesse","d[X/H]"
]].copy()

# ===================== 4_ Write outputs =====================

# ---- (A) Raw per-line table for inspection/debugging ----
raw_path = output_abundances_result_path_linebyline_q2.replace(".csv", "_raw.csv")
df_merge_out.to_csv(raw_path, index=False)
print("Saved raw per-line:", raw_path)

# ---- (B) Summary table for step 5_: aggregate by element×ion ----
ROMAN2INT = {"I":1, "II":2, "III":3, "IV":4}

def to_num_ion_token(s):
    """Convert 'Fe I' -> 'Fe 1', 'Ti II' -> 'Ti 2', 'C' -> 'C 1'."""
    parts = str(s).strip().split()
    if len(parts) == 1:
        return f"{parts[0]} 1"
    base, ion = parts[0], parts[1]
    ion_u = ion.upper()
    if ion_u in ROMAN2INT:
        return f"{base} {ROMAN2INT[ion_u]}"
    try:
        return f"{base} {int(float(ion))}"
    except:
        return f"{base} 1"

df_for_plot = df_merge_out.copy()
df_for_plot["element"] = df_for_plot["element"].apply(to_num_ion_token)

# Use d[X/H] as the line-by-line differential [X/H]
g = df_for_plot.groupby("element")["d[X/H]"]
df_summary = g.agg(
    n_lines="count",
    **{"[X/H]":"mean"}
).reset_index()

# Compute standard deviation (ddof=1)
std = g.std(ddof=1).reset_index(drop=True)
df_summary["std_[X/H]"] = std

# Sort with Fe 1, Fe 2 first
order = []
for x in ["Fe 1", "Fe 2"]:
    if x in df_summary["element"].values:
        order.append(x)
others = [x for x in df_summary["element"].tolist() if x not in order]
df_summary["__ord"] = pd.Categorical(df_summary["element"], categories=order+sorted(others), ordered=True)
df_summary = df_summary.sort_values("__ord").drop(columns="__ord").reset_index(drop=True)

# Overwrite the summary file for step 5_
df_summary.to_csv(output_abundances_result_path_linebyline_q2, index=False)
print("Saved summary for 5_:", output_abundances_result_path_linebyline_q2)

print(df_summary.head(10))